In [ ]:
# 01  Filtering and Analysis- EMPATICA
 
 # import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


from scipy.signal import find_peaks
from scipy.signal import butter, filtfilt



# load data
data = pd.read_csv("../Data/raw/WESAD/S2/S2_E4_Data/BVP.csv", header=None, names=["BVP"])
 
bvp = data['BVP'].iloc[2:].astype(float).values       # Skip the first two rows as they contain metadata rather than physiological samples.

# ===================================
# Butterworth Bandpass Filtering
# ===================================

# Sampling frequency
fs = 64

# Bandpass range
lowcut = 0.7
highcut = 5

nyquist = fs / 2

b, a = butter(
    4,
    [lowcut/nyquist, highcut/nyquist],
    btype='band'
)

bvp_filtered = filtfilt(b, a, bvp)


plt.figure(figsize=(12,5))

samples = 2000

plt.plot(bvp[:samples], label='Raw BVP', alpha=0.7)
plt.plot(bvp_filtered[:samples], label='Filtered BVP')

plt.title('Raw vs Filtered BVP')
plt.xlabel('Sample Index')
plt.ylabel('BVP Amplitude')
plt.legend()
plt.grid(True)

plt.show()

 

start = 0
end = 3000

 
plt.plot(bvp_filtered[start:end], label='Filtered BVP')


plt.title("Filtered BVP")
plt.xlabel("Samples")
plt.ylabel("Amplitude")

plt.grid(True)
plt.show()


raw_z = np.abs((bvp - np.mean(bvp))/np.std(bvp))
filtered_z = np.abs((bvp_filtered - np.mean(bvp_filtered))/np.std(bvp_filtered))

raw_spikes = np.sum(raw_z > 3)
filtered_spikes = np.sum(filtered_z > 3)

noise_raw = np.std(np.diff(bvp))
noise_filtered = np.std(np.diff(bvp_filtered))


# Raw BVP peaks
raw_peaks, _ = find_peaks(
    bvp,  
    prominence=0.1,      # may need tuning
    distance=32          # ~0.5 sec at 64 Hz
)

# Filtered BVP peaks
filtered_peaks, _ = find_peaks(
    bvp_filtered,
    prominence=0.1,
    distance=32
)

 
results = pd.DataFrame({
    "Metric":["Mean","Std","Spikes (>3Ïƒ)","Noise Estimate", "Peak Count"],
    "Raw":[
        np.mean(bvp),
        np.std(bvp),
        raw_spikes,
        noise_raw,
        len(raw_peaks)
    ],
    "Filtered":[
        np.mean(bvp_filtered),
        np.std(bvp_filtered),
        filtered_spikes,
        noise_filtered,
        len(filtered_peaks)
    ]
})

print(results)
print("Min (Raw):", np.min(bvp))
print("Max (Raw):", np.max(bvp))

print("Min (Filtered):", np.min(bvp_filtered))
print("Max (Filtered):", np.max(bvp_filtered))
 

In [ ]:
#02  EDA- EMPATICA

# Imports
import pandas as pd
import matplotlib.pyplot as plt
import scipy.signal
import numpy as np
from scipy.signal import find_peaks
from scipy.stats import linregress
from scipy.signal import butter, filtfilt, medfilt
# load data
data = pd.read_csv("../Data/raw/WESAD/S2/S2_E4_Data/EDA.csv", header=None, names=["EDA"])
eda_raw = data['EDA'].iloc[2:].astype(float).values 

# ------------LOW PASS FILTERING------------
# filter
def lowpass_filter(signal, cutoff, fs, order=4):
    nyquist = fs / 2
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='low')
    return filtfilt(b, a, signal)
eda_filtered = lowpass_filter(
    eda_raw,
    cutoff=1,
    fs=4
)


# visulaize
plt.figure(figsize=(12,4))
plt.plot(eda_filtered)
plt.title("Filtered EDA")
plt.xlabel("Samples")
plt.ylabel("EDA")
plt.show()

# Compare them on the same graph
plt.figure(figsize=(12,4))
plt.plot(eda_raw, label='Raw EDA', alpha=0.6)
plt.plot(eda_filtered, label='Filtered EDA', linewidth=2)
plt.xlabel("Samples")
plt.ylabel("EDA")
plt.legend()
plt.title("Raw vs Filtered EDA")
plt.show()

# Plot only a portion:
plt.figure(figsize=(12,4))

plt.plot(eda_raw[:1000], label='Raw EDA')
plt.plot(eda_filtered[:1000], label='Filtered EDA')
plt.xlabel("Samples")
plt.ylabel("EDA")
plt.legend()
plt.show()



# quality metrics
# -------------------------
# Noise
# -------------------------
def calculate_noise(signal):
    return np.std(np.diff(signal))

# -------------------------
# Spikes
# -------------------------
def count_spikes(signal, threshold=3):
    diff_signal = np.abs(np.diff(signal))
    spike_threshold = threshold * np.std(diff_signal)
    return np.sum(diff_signal > spike_threshold)

# -------------------------
# Peaks (EDA-specific)
# -------------------------
def count_peaks(signal, prominence=0.02):
    peaks, _ = find_peaks(signal, prominence=prominence)
    return len(peaks)

# -------------------------
# Drift
# -------------------------
def calculate_drift(signal):
    x = np.arange(len(signal))
    slope, _, _, _, _ = linregress(x, signal)
    return slope

# -------------------------
# Metrics Function
# -------------------------
def get_metrics(signal):
    return {
        "Samples": len(signal),
        "Mean": np.mean(signal),
        "Std": np.std(signal),
        "Noise": calculate_noise(signal),
        "Spikes": count_spikes(signal),
        "Peaks": count_peaks(signal),
        "Drift": calculate_drift(signal)
    }

# -------------------------
# Calculate Metrics
# -------------------------
raw_metrics = get_metrics(eda_raw)
filtered_metrics = get_metrics(eda_filtered)

# -------------------------
# Print Results
# -------------------------
print("========== RAW EDA ==========")
for key, value in raw_metrics.items():
    print(f"{key:<10}: {value}")

print("\n========== FILTERED EDA ==========")
for key, value in filtered_metrics.items():
    print(f"{key:<10}: {value}")

In [ ]:
# 03 HR EMPATICA

# IMPORTS
import pandas as pd
import numpy as np
import numpy as np
from scipy.signal import savgol_filter
from scipy.stats import zscore
from scipy.signal import find_peaks
from scipy.stats import linregress
import matplotlib.pyplot as plt


# load data
hr_df = pd.read_csv("../Data/raw/WESAD/S2/S2_E4_Data/HR.csv", header=None)

fs = float(hr_df.iloc[1,0])      # should be 1 Hz
hr = hr_df.iloc[2:,0].astype(float).values




# ---------------------------------
# FILTER HR
# ---------------------------------

hr_filtered = savgol_filter(
    hr,
    window_length=11,  # must be odd
    polyorder=2
)

# ---------------------------------
# QUALITY METRICS
# ---------------------------------

def calculate_noise(signal):
    return np.std(np.diff(signal))

def count_spikes(signal, threshold=3):
    z = np.abs(zscore(signal))
    return np.sum(z > threshold)

def calculate_drift(signal):
    x = np.arange(len(signal))
    slope, _, _, _, _ = linregress(x, signal)
    return slope

# ---------------------------------
# RAW METRICS
# ---------------------------------

raw_mean = np.mean(hr)
raw_std = np.std(hr)
raw_noise = calculate_noise(hr)
raw_spikes = count_spikes(hr)
raw_drift = calculate_drift(hr)

# ---------------------------------
# FILTERED METRICS
# ---------------------------------

filt_mean = np.mean(hr_filtered)
filt_std = np.std(hr_filtered)
filt_noise = calculate_noise(hr_filtered)
filt_spikes = count_spikes(hr_filtered)
filt_drift = calculate_drift(hr_filtered)

# ---------------------------------
# RESULTS
# ---------------------------------

print("========== RAW HR ==========")
print("Mean      :", raw_mean)
print("Std       :", raw_std)
print("Spikes    :", raw_spikes)
print("Noise     :", raw_noise)
print("Drift     :", raw_drift)

print("\n========== FILTERED HR ==========")
print("Mean      :", filt_mean)
print("Std       :", filt_std)
print("Spikes    :", filt_spikes)
print("Noise     :", filt_noise)
print("Drift     :", filt_drift)

print("\n========== IMPROVEMENT ==========")
print("Noise Reduction (%) :",
      ((raw_noise - filt_noise)/raw_noise)*100)

print("Std Reduction (%) :",
      ((raw_std - filt_std)/raw_std)*100)

# ---------------------------------
# PLOT
# ---------------------------------

import matplotlib.pyplot as plt

plt.figure(figsize=(12,5))

plt.plot(hr, label="Raw HR", alpha=0.5)

plt.plot(
    hr_filtered,
    label="Filtered HR",
    linewidth=2
)

plt.title("Raw vs Filtered HR")
plt.xlabel("Sample Index")
plt.ylabel("Heart Rate (bpm)")

plt.legend()
plt.grid()

plt.show()

In [ ]:
# 04 IBI EMPATICA

# import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import zscore

# =========================
# 1. LOAD & CLEAN IBI
# =========================
file_path = "../Data/raw/WESAD/S2/S2_E4_Data/IBI.csv"

df = pd.read_csv(file_path, sep="\t", header=None, comment="#")

df = df[0].str.split(",", expand=True)
df.columns = ["timestamp", "ibi"]

df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce")
df["ibi"] = pd.to_numeric(df["ibi"], errors="coerce")

df = df.dropna()
df = df[(df["ibi"] >= 0.3) & (df["ibi"] <= 2.0)]
df = df.sort_values("timestamp").reset_index(drop=True)

raw_ibi = df["ibi"].values

# =========================
# 2. INTERPOLATION
# =========================
ibi_interp = pd.Series(raw_ibi).interpolate().bfill().ffill().values

# =========================
# 3. SMOOTHING (MEDIAN FILTER)
# =========================
ibi_filtered = pd.Series(ibi_interp).rolling(
    window=5, center=True, min_periods=1
).median().values

# =========================
# 4. ALIGN
# =========================
n = min(len(raw_ibi), len(ibi_filtered))

raw = raw_ibi[:n]
filtered = ibi_filtered[:n]

# =========================
# 5. QUALITY METRICS
# =========================
noise_raw = np.std(np.diff(raw))
noise_filtered = np.std(np.diff(filtered))

spikes_raw = np.sum(np.abs(zscore(raw)) > 3)
spikes_filtered = np.sum(np.abs(zscore(filtered)) > 3)

smooth_raw = np.mean(np.abs(np.diff(raw)))
smooth_filtered = np.mean(np.abs(np.diff(filtered)))

mean_raw = np.mean(raw)
mean_filtered = np.mean(filtered)

std_raw = np.std(raw)
std_filtered = np.std(filtered)

outliers_raw = spikes_raw
outliers_filtered = spikes_filtered

cv_raw = std_raw / mean_raw
cv_filtered = std_filtered / mean_filtered

print("\n===== IBI QUALITY METRICS =====\n")

print(f"Mean IBI       | Raw: {mean_raw:.6f} | Filtered: {mean_filtered:.6f}")
print(f"Std IBI        | Raw: {std_raw:.6f} | Filtered: {std_filtered:.6f}")
print(f"Outliers       | Raw: {outliers_raw} | Filtered: {outliers_filtered}")
print(f"Noise          | Raw: {noise_raw:.6f} | Filtered: {noise_filtered:.6f}")
print(f"Smoothness     | Raw: {smooth_raw:.6f} | Filtered: {smooth_filtered:.6f}")
print(f"Consistency    | Raw: {cv_raw:.6f} | Filtered: {cv_filtered:.6f}")

# =========================
# 6. PLOT 1: FULL COMPARISON
# =========================
plt.figure(figsize=(14,5))
plt.plot(raw[:400], label="Raw IBI", alpha=0.6)
plt.plot(filtered[:400], label="Filtered IBI", linewidth=2)

plt.title("IBI Signal: Raw vs Filtered (First 400 Beats)")
plt.xlabel("Beat Index")
plt.ylabel("IBI (seconds)")
plt.legend()
plt.grid()
plt.show()

# =========================
# 7. PLOT 2: ZOOMED VIEW
# =========================
start = 100
end = 200

plt.figure(figsize=(14,5))
plt.plot(raw[start:end], label="Raw IBI")
plt.plot(filtered[start:end], label="Filtered IBI", linewidth=2)

plt.title("IBI Signal: Zoomed View (100â€“200 Beats)")
plt.xlabel("Beat Index")
plt.ylabel("IBI (seconds)")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# 05 TEMPERATURE- E4


# imports

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
from scipy.stats import linregress

# =========================
# 1. LOAD DATA
# =========================
file_path = "../Data/raw/WESAD/S2/S2_E4_Data/TEMP.csv"

data = pd.read_csv(file_path, header=None)

temp = data[0].iloc[1:].astype(float).values
temp = temp[5:]   # remove initial noise

# =========================
# 2. FILTER FUNCTION
# =========================
def lowpass_filter(signal, cutoff=0.5, fs=4, order=4):
    nyq = fs / 2
    b, a = butter(order, cutoff / nyq, btype='low')
    return filtfilt(b, a, signal)

filtered_temp = lowpass_filter(temp)

# =========================
# 3. METRICS FUNCTION
# =========================
def get_metrics(signal):
    return {
        "Mean": np.mean(signal),
        "Std": np.std(signal),
        "Min": np.min(signal),
        "Max": np.max(signal),
        "Range": np.max(signal) - np.min(signal),
        "Noise": np.std(np.diff(signal)),
        "Drift": linregress(np.arange(len(signal)), signal).slope
    }

# =========================
# 4. COMPARE METRICS
# =========================
raw_metrics = get_metrics(temp)
filt_metrics = get_metrics(filtered_temp)

print("\n===== TEMPERATURE METRICS =====\n")

for k in raw_metrics:
    print(f"{k:<10} | Raw: {raw_metrics[k]:.6f} | Filtered: {filt_metrics[k]:.6f}")

# =========================
# 5. PERCENT CHANGE
# =========================
print("\n===== PERCENT CHANGE =====\n")

for k in raw_metrics:
    if raw_metrics[k] != 0:
        change = ((filt_metrics[k] - raw_metrics[k]) / raw_metrics[k]) * 100
        print(f"{k:<10}: {change:.2f}%")

# =========================
# 6. VISUALIZATION
# =========================
plt.figure(figsize=(14,5))
plt.plot(temp[:500], label="Raw TEMP", alpha=0.6)
plt.plot(filtered_temp[:500], label="Filtered TEMP", linewidth=2)

plt.title("Temperature: Raw vs Filtered")
plt.xlabel("Samples")
plt.ylabel("Temperature (Â°C)")
plt.legend()
plt.grid()
plt.show()

# Zoomed plot
plt.figure(figsize=(14,5))
plt.plot(temp[100:200], label="Raw TEMP")
plt.plot(filtered_temp[100:200], label="Filtered TEMP", linewidth=2)

plt.title("Temperature Zoomed View")
plt.xlabel("Samples")
plt.ylabel("Temperature (Â°C)")
plt.legend()
plt.grid()
plt.show()

RESPIBAN

In [ ]:
# 01 ECG---RESPIBAN

# imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt

# load data
file_path =  "../Data/raw/WESAD/S2/S2_respiban.txt"

data = pd.read_csv(
    file_path,
    sep="\t",
    comment="#",        # ignores JSON header + metadata
    header=None,
    engine="python"
)
ecg = data.iloc[:, 2].values   # CH1 as ECG (change if needed)

# -----------------------------
# STEP 2: Bandpass Filter (0.5â€“40 Hz)
# -----------------------------
def bandpass_filter(signal, fs=700, low=0.5, high=40, order=2):
    nyq = 0.5 * fs
    b, a = butter(order, [low/nyq, high/nyq], btype='band')
    return filtfilt(b, a, signal)

ecg_filtered = bandpass_filter(ecg)

# -----------------------------
# STEP 3: Plot RAW ECG
# -----------------------------
plt.figure(figsize=(14,4))
plt.plot(ecg[:5000])
plt.title("Raw ECG Signal (Zoomed)")
plt.xlabel("Samples")
plt.ylabel("Amplitude")
plt.grid()
plt.show()

# -----------------------------
# STEP 4: Plot FILTERED ECG
# -----------------------------
plt.figure(figsize=(14,4))
plt.plot(ecg_filtered[:5000])
plt.title("Filtered ECG Signal (0.5â€“40 Hz)")
plt.xlabel("Samples")
plt.ylabel("Amplitude")
plt.grid()
plt.show()

# -----------------------------
# STEP 5: COMPARISON PLOT
# -----------------------------
plt.figure(figsize=(14,5))
plt.plot(ecg[:3000], label="Raw ECG", alpha=0.6)
plt.plot(ecg_filtered[:3000], label="Filtered ECG", linewidth=2)
plt.title("Raw vs Filtered ECG Comparison")
plt.xlabel("Samples")
plt.ylabel("Amplitude")
plt.legend()
plt.grid()
plt.show()

ecg_centered = ecg - np.mean(ecg)  #dc offset removal
 

n_samples = 3000

plt.figure(figsize=(12,5))

plt.plot(ecg_centered[:n_samples], label="Raw ECG (DC Offset Removed)")
plt.plot(ecg_filtered[:n_samples], label="Filtered ECG")

plt.title("ECG: DC Offset Removed vs Filtered")
plt.xlabel("Samples")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True)

plt.show()
start = 10000
n_samples = 1000

plt.figure(figsize=(12,5))

plt.plot(ecg_centered[start:start+n_samples], label="Raw ECG (DC Removed)")
plt.plot(ecg_filtered[start:start+n_samples], label="Filtered ECG")

plt.title("ECG Comparison (Zoomed)")
plt.xlabel("Samples")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True)

plt.show()

import numpy as np
import pandas as pd
from scipy.signal import find_peaks

# ----------------------------
# Remove DC Offset
# ----------------------------
ecg_centered = ecg - np.mean(ecg)

# ----------------------------
# Mean
# ----------------------------
ecg_mean_raw = np.mean(ecg)
ecg_mean_filtered = np.mean(ecg_filtered)
ecg_mean_centered = np.mean(ecg_centered)

# ----------------------------
# Standard Deviation
# ----------------------------
ecg_std_raw = np.std(ecg)
ecg_std_filtered = np.std(ecg_filtered)
ecg_std_centered = np.std(ecg_centered)

# ----------------------------
# Noise Estimate
# ----------------------------
ecg_noise_raw = np.std(np.diff(ecg))
ecg_noise_filtered = np.std(np.diff(ecg_filtered))
ecg_noise_centered = np.std(np.diff(ecg_centered))

# ----------------------------
# Baseline Drift
# ----------------------------
window = 7000

ecg_drift_raw = np.std(
    np.convolve(ecg, np.ones(window)/window, mode='same')
)

ecg_drift_filtered = np.std(
    np.convolve(ecg_filtered,
                np.ones(window)/window,
                mode='same')
)

ecg_drift_centered = np.std(
    np.convolve(ecg_centered,
                np.ones(window)/window,
                mode='same')
)

# ----------------------------
# Outlier Count (>3Ïƒ)
# ----------------------------
z_raw = np.abs((ecg - ecg_mean_raw) / ecg_std_raw)
z_filtered = np.abs((ecg_filtered - ecg_mean_filtered) / ecg_std_filtered)
z_centered = np.abs((ecg_centered - ecg_mean_centered) / ecg_std_centered)

outliers_raw = np.sum(z_raw > 3)
outliers_filtered = np.sum(z_filtered > 3)
outliers_centered = np.sum(z_centered > 3)

# ----------------------------
# R Peak Detection
# ----------------------------
fs = 700

peaks_raw, _ = find_peaks(
    ecg_centered,
    distance=0.4*fs,
    prominence=np.std(ecg_centered)
)

peaks_filtered, _ = find_peaks(
    ecg_filtered,
    distance=0.4*fs,
    prominence=np.std(ecg_filtered)
)

# ----------------------------
# RR Interval
# ----------------------------
rr_raw = np.diff(peaks_raw) / fs
rr_filtered = np.diff(peaks_filtered) / fs

rr_mean_raw = np.mean(rr_raw)
rr_mean_filtered = np.mean(rr_filtered)

rr_std_raw = np.std(rr_raw)
rr_std_filtered = np.std(rr_filtered)

# ----------------------------
# Heart Rate
# ----------------------------
hr_raw = 60 / rr_raw
hr_filtered = 60 / rr_filtered

hr_mean_raw = np.mean(hr_raw)
hr_mean_filtered = np.mean(hr_filtered)

hr_std_raw = np.std(hr_raw)
hr_std_filtered = np.std(hr_filtered)

# ----------------------------
# Results Table
# ----------------------------
results = pd.DataFrame({

    "Metric":[
        "Mean",
        "Std",
        "Noise Estimate",
        "Baseline Drift",
        "Outliers (>3Ïƒ)",
        "R Peak Count",
        "Mean RR Interval (s)",
        "RR Interval Std (s)",
        "Mean Heart Rate (BPM)",
        "Heart Rate Std (BPM)"
    ],

    "Raw":[
        ecg_mean_raw,
        ecg_std_raw,
        ecg_noise_raw,
        ecg_drift_raw,
        outliers_raw,
        len(peaks_raw),
        rr_mean_raw,
        rr_std_raw,
        hr_mean_raw,
        hr_std_raw
    ],

    "Filtered":[
        ecg_mean_filtered,
        ecg_std_filtered,
        ecg_noise_filtered,
        ecg_drift_filtered,
        outliers_filtered,
        len(peaks_filtered),
        rr_mean_filtered,
        rr_std_filtered,
        hr_mean_filtered,
        hr_std_filtered
    ],

    "DC Offset Removed":[
        ecg_mean_centered,
        ecg_std_centered,
        ecg_noise_centered,
        ecg_drift_centered,
        outliers_centered,
        len(peaks_raw),
        rr_mean_raw,
        rr_std_raw,
        hr_mean_raw,
        hr_std_raw
    ]

})

print(results)

In [ ]:
# 02 EDA----RESPIBAN
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, find_peaks


# ///load data
file_path =  "../Data/raw/WESAD/S2/S2_respiban.txt"

data = pd.read_csv(
    file_path,
    sep="\t",
    comment="#",        # ignores JSON header + metadata
    header=None,
    engine="python"
)
EDA = data.iloc[:,3].values     #load eda

EDA = data.iloc[:,3].values         #remove dc offset

fs = 700
cutoff = 5
order = 4

nyquist = fs / 2

b, a = butter(
    order,
    cutoff / nyquist,
    btype="low"
)

EDA_filtered = filtfilt(b, a, EDA)

plt.figure(figsize=(15,4))

plt.plot(EDA_filtered,color="orange")

plt.title("Filtered Chest EDA")
plt.xlabel("Samples")
plt.ylabel("Amplitude")

plt.grid(True)

plt.show()

# raw vs filtered
plt.figure(figsize=(15,5))

plt.plot(EDA,
         label="Raw")

plt.plot(EDA_filtered,
         label="Filtered")

plt.title("Raw vs Filtered Chest EDA")

plt.xlabel("Samples")
plt.ylabel("Amplitude")

plt.legend()
plt.grid(True)

plt.show()
 

EDA = np.array(EDA)
EDA_filtered = np.array(EDA_filtered)
start = 1000        # change this
end = 1500          # small segment for zoom

# -----------------------------
# Slice signals
# -----------------------------
raw_zoom = EDA[start:end]
filtered_zoom = EDA_filtered[start:end]

# -----------------------------
# Plot zoomed comparison
# -----------------------------
plt.figure(figsize=(14, 5))

plt.plot(raw_zoom, label="Raw EDA", alpha=0.6)
plt.plot(filtered_zoom, label="Filtered EDA", alpha=0.9)

plt.title("Zoomed EDA Comparison (Raw vs Filtered)")
plt.xlabel("Sample Index (Zoomed Window)")
plt.ylabel("EDA Amplitude")
plt.legend()
plt.grid()

plt.show()


# -----------------------------
# METRIC FUNCTIONS
# -----------------------------
def noise_estimate(signal):
    return np.std(np.diff(signal))

def baseline_drift(signal):
    return np.mean(signal) - np.median(signal)

def outlier_count(signal):
    mean = np.mean(signal)
    std = np.std(signal)
    if std == 0:
        return 0
    z = (signal - mean) / std
    return np.sum(np.abs(z) > 3)

def scr_peaks(signal, fs=4):
    peaks, _ = find_peaks(signal, prominence=0.01, distance=int(fs*1))
    return len(peaks)

def slope_variability(signal):
    return np.std(np.diff(signal))

# -----------------------------
# FULL METRIC REPORT FUNCTION
# -----------------------------
def compute_metrics(signal, fs=4):
    return {
        "Mean": np.mean(signal),
        "Std": np.std(signal),
        "Noise Estimate": noise_estimate(signal),
        "Baseline Drift": baseline_drift(signal),
        "Outliers (>3Ïƒ)": outlier_count(signal),
        "SCR Peaks": scr_peaks(signal, fs),
        "Slope Variability": slope_variability(signal)
    }

# -----------------------------
# LOAD YOUR SIGNALS
# -----------------------------
# Replace with your actual arrays
# eda_raw = ...
# eda_filtered = ...

raw_metrics = compute_metrics(EDA)
filtered_metrics = compute_metrics(EDA_filtered)

# -----------------------------
# CREATE COMPARISON TABLE
# -----------------------------
rows = []

for key in raw_metrics.keys():
    raw_val = raw_metrics[key]
    filt_val = filtered_metrics[key]

    # % change
    if raw_val != 0:
        pct_change = ((filt_val - raw_val) / raw_val) * 100
    else:
        pct_change = np.nan

    rows.append([key, raw_val, filt_val, pct_change])

df = pd.DataFrame(rows, columns=[
    "Metric",
    "Raw EDA",
    "Filtered EDA",
    "% Change"
])

# -----------------------------
# DISPLAY TABLE
# -----------------------------
pd.set_option("display.float_format", "{:.4f}".format)
print(df)

 

In [ ]:
# 03- EMG- RESPIBAN

# imports
import pandas as pd
from scipy.signal import butter, filtfilt
import numpy as np

# data load
respiban = pd.read_csv(
    "../Data/raw/WESAD/S2/S2_respiban.txt",
    sep="\t",
    comment="#",
    header=None
)
emg = respiban.iloc[:, 4].values   # or respiban[3].values

# filter
fs = 700
lowcut = 20
highcut = 250

nyquist = fs / 2

b, a = butter(
    4,
    [lowcut/nyquist, highcut/nyquist],
    btype='band'
)

emg_filtered = filtfilt(b, a, emg)
emg_centered = emg - np.mean(emg)                       #dc offset removed
 
# visualisation
plt.figure(figsize=(12,4))

plt.plot(emg[:5000], label="Raw EMG")
plt.plot(emg_centered[:5000], label="Raw EMG (DC Removed)")

plt.title("Effect of Removing DC Offset")
plt.xlabel("Sample Index")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12,4))

plt.plot(emg_centered[:5000], label="Raw (DC Removed)")
plt.plot(emg_filtered[:5000], label="Filtered")

plt.title("DC Removed vs Filtered EMG")
plt.xlabel("Sample Index")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True)
plt.show()

 
# Number of samples to display
n_samples = 300

plt.figure(figsize=(12,4))

plt.plot(emg_centered[:n_samples], label="Raw (DC Removed)", linewidth=1.5)
plt.plot(emg_filtered[:n_samples], label="Filtered", linewidth=1.5)

plt.title(f"Raw (DC Removed) vs Filtered EMG ({n_samples} Samples)")
plt.xlabel("Sample Index")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True)

plt.show()
 
# quality metrics



emg_centered = emg - np.mean(emg)
# Mean
emg_mean_raw = np.mean(emg)
emg_mean_filtered = np.mean(emg_filtered)
emg_mean_centered = np.mean(emg_centered)
# Standard deviation
emg_std_raw = np.std(emg)
emg_std_filtered = np.std(emg_filtered)
emg_std_centered =np.std(emg_centered)

# Spikes
emg_z_raw = np.abs((emg - emg_mean_raw) / emg_std_raw)
emg_z_filtered = np.abs((emg_filtered - emg_mean_filtered) / emg_std_filtered)
emg_z_centered = np.abs(
    (emg_centered - emg_mean_centered) / emg_std_centered
)

emg_spikes_raw = np.sum(emg_z_raw > 3)
emg_spikes_filtered = np.sum(emg_z_filtered > 3)
emg_spikes_centered = np.sum(emg_z_centered > 3)

# Noise estimate
emg_noise_raw = np.std(np.diff(emg))
emg_noise_filtered = np.std(np.diff(emg_filtered))
emg_noise_centered = np.std(np.diff(emg_centered))
# Drift estimate
window = 7000

emg_drift_raw = np.std(
    np.convolve(emg, np.ones(window)/window, mode='same')
)

emg_drift_filtered = np.std(
    np.convolve(emg_filtered, np.ones(window)/window, mode='same')
)
emg_drift_centered = np.std(
    np.convolve(emg_centered,
                np.ones(window)/window,
                mode='same')
)

results = pd.DataFrame({
    "Metric": ["Mean","Std","Spikes (>3Ïƒ)","Noise Estimate","Drift Estimate"],
    "Raw": [
        emg_mean_raw,
        emg_std_raw,
        emg_spikes_raw,
        emg_noise_raw,
        emg_drift_raw
    ],
    "Filtered": [
        emg_mean_filtered,
        emg_std_filtered,
        emg_spikes_filtered,
        emg_noise_filtered,
        emg_drift_filtered
    ],

    "Centered(DC offset removed)":[
        emg_mean_centered,
        emg_std_centered,
        emg_spikes_centered,
        emg_noise_centered,
        emg_drift_centered
    ]
})

print(results)

In [ ]:
# 04 Respiration----respiban

# imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt
from scipy.signal import butter, filtfilt, find_peaks

# data load
file_path = "../Data/raw/WESAD/S2/S2_respiban.txt"

respiban = pd.read_csv(
    file_path,
    sep="\t",        # change to " " if space-separated
    header=None,
    comment="#",     # ignores metadata lines starting with #
)

RESPIRATION = respiban.iloc[:,9].values

# filter
fs= 700
lowcut = 0.05
highcut = 2.0
order = 4
nyquist = fs / 2

sos = butter(
    4,
    [lowcut/(fs/2), highcut/(fs/2)],
    btype='band',
    output='sos'
)

RESPIRATION_filtered = sosfiltfilt(sos, RESPIRATION)

 
# visualisation
plt.figure(figsize=(15,5))

plt.plot(RESPIRATION,
         label="Raw",
         linewidth=1)

plt.plot(RESPIRATION_filtered,
         label="Filtered",
         linewidth=1)

plt.title("Raw vs Filtered Respiration")

plt.xlabel("Samples")
plt.ylabel("Amplitude")

plt.legend()

plt.grid(True)

plt.show()
samples = 3000

plt.figure(figsize=(15,5))

plt.plot(RESPIRATION[:samples],
         label="Raw",
         linewidth=1.5)

plt.plot(RESPIRATION_filtered[:samples],
         label="Filtered",
         linewidth=1.5)

plt.title("Raw vs Filtered Respiration (First 3000 Samples)")

plt.xlabel("Samples")
plt.ylabel("Amplitude")

plt.legend()

plt.grid(True)

plt.show()
print(respiban.iloc[:5, :10])
seconds = 20
samples = seconds * fs

plt.figure(figsize=(15,4))
plt.plot(RESPIRATION[:samples])
plt.title("Raw Respiration (First 20 Seconds)")
plt.xlabel("Samples")
plt.ylabel("Amplitude")
plt.grid(True)
plt.show()

# Remove DC offset from raw respiration
RESPIRATION_centered = RESPIRATION - np.mean(RESPIRATION)

# Plot first 3000 samples
samples = 3000

plt.figure(figsize=(15,5))

plt.plot(
    RESPIRATION_centered[:samples],
    label="Raw (DC Offset Removed)",
    linewidth=1.5
)

plt.plot(
    RESPIRATION_filtered[:samples],
    label="Filtered",
    linewidth=1.5
)

plt.title("Raw (DC Offset Removed) vs Filtered Respiration")
plt.xlabel("Samples")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True)

plt.show()

# quality metrics
import numpy as np
import pandas as pd
from scipy.signal import find_peaks

# ---------------------------------
# Remove DC Offset
# ---------------------------------
RESPIRATION_centered = RESPIRATION - np.mean(RESPIRATION)

# ---------------------------------
# Mean
# ---------------------------------
mean_centered = np.mean(RESPIRATION_centered)
mean_filtered = np.mean(RESPIRATION_filtered)

# ---------------------------------
# Standard Deviation
# ---------------------------------
std_centered = np.std(RESPIRATION_centered)
std_filtered = np.std(RESPIRATION_filtered)

# ---------------------------------
# Noise Estimate
# ---------------------------------
noise_centered = np.std(np.diff(RESPIRATION_centered))
noise_filtered = np.std(np.diff(RESPIRATION_filtered))

# ---------------------------------
# Baseline Drift
# ---------------------------------
window = 7000

drift_centered = np.std(
    np.convolve(
        RESPIRATION_centered,
        np.ones(window)/window,
        mode='same'
    )
)

drift_filtered = np.std(
    np.convolve(
        RESPIRATION_filtered,
        np.ones(window)/window,
        mode='same'
    )
)

# ---------------------------------
# Outlier Count (>3Ïƒ)
# ---------------------------------
z_centered = np.abs(
    (RESPIRATION_centered - mean_centered) / std_centered
)

z_filtered = np.abs(
    (RESPIRATION_filtered - mean_filtered) / std_filtered
)

outliers_centered = np.sum(z_centered > 3)
outliers_filtered = np.sum(z_filtered > 3)

# ---------------------------------
# Breath Peak Detection
# ---------------------------------
peaks_centered, _ = find_peaks(
    RESPIRATION_centered,
    distance=fs * 1.5,
    prominence=0.3 * std_centered
)

peaks_filtered, _ = find_peaks(
    RESPIRATION_filtered,
    distance=fs * 1.5,
    prominence=0.3 * std_filtered
)

# ---------------------------------
# Breath Intervals
# ---------------------------------
interval_centered = np.diff(peaks_centered) / fs
interval_filtered = np.diff(peaks_filtered) / fs

# ---------------------------------
# Breathing Rate
# ---------------------------------
duration = len(RESPIRATION) / fs

breathing_rate_centered = len(peaks_centered) / (duration / 60)
breathing_rate_filtered = len(peaks_filtered) / (duration / 60)

# ---------------------------------
# Results Table
# ---------------------------------
results = pd.DataFrame({

    "Metric":[
        "Mean",
        "Std",
        "Noise Estimate",
        "Baseline Drift",
        "Outliers (>3Ïƒ)",
        "Breath Count",
        "Mean Breath Interval (s)",
        "Breath Interval Std (s)",
        "Breathing Rate (breaths/min)"
    ],

    "DC Offset Removed":[
        mean_centered,
        std_centered,
        noise_centered,
        drift_centered,
        outliers_centered,
        len(peaks_centered),
        np.mean(interval_centered),
        np.std(interval_centered),
        breathing_rate_centered
    ],

    "Filtered":[
        mean_filtered,
        std_filtered,
        noise_filtered,
        drift_filtered,
        outliers_filtered,
        len(peaks_filtered),
        np.mean(interval_filtered),
        np.std(interval_filtered),
        breathing_rate_filtered
    ]
})

print(results)

In [ ]:
# 05 Temperature----respiban


# imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, find_peaks

# load data
file_path = "../Data/raw/WESAD/S2/S2_respiban.txt"

data = pd.read_csv(
    file_path,
    sep="\t",        # change to " " if space-separated
    header=None,
    comment="#",     # ignores metadata lines starting with #
)
 

# =========================================================
# 2. EXTRACT TEMPERATURE (INDEX 5)
# =========================================================
temp_raw = data.iloc[:, 5].values.astype(float)

# =========================================================
# 3. DC OFFSET REMOVAL (OPTIONAL BUT GOOD)
# =========================================================
temp_dc = temp_raw - np.mean(temp_raw)

# =========================================================
# 4. LOW-PASS FILTER (TEMPERATURE = SLOW SIGNAL)
# =========================================================
def lowpass(signal, fs=1, cutoff=0.3):
    nyq = 0.5 * fs
    b, a = butter(3, cutoff / nyq, btype='low')
    return filtfilt(b, a, signal)

temperature_filtered = lowpass(temp_dc, fs=1, cutoff=0.3)

# =========================================================
# 5. QUALITY METRICS
# =========================================================
def metrics(x):
    return {
        "Mean": np.mean(x),
        "Std": np.std(x),
        "Noise": np.std(np.diff(x)),
        "Drift": np.mean(x) - np.median(x),
        "Outliers": np.sum(np.abs((x - np.mean(x)) / (np.std(x)+1e-6)) > 3),
        "Slope Var": np.std(np.diff(x))
    }

raw_m = metrics(temp_dc)
filt_m = metrics(temperature_filtered)

# =========================================================
# 6. TABLE
# =========================================================
rows = []
for k in raw_m.keys():
    pct = ((filt_m[k] - raw_m[k]) / (raw_m[k] + 1e-6)) * 100
    rows.append([k, raw_m[k], filt_m[k], pct])

temp_table = pd.DataFrame(rows, columns=["Metric", "Raw", "Filtered", "% Change"])

print(temp_table)

# =========================================================
# 7. PLOT (RAW vs FILTERED - ZOOMED)
# =========================================================
plt.figure(figsize=(14,4))

plt.plot(temp_dc[:1000], label="Raw (DC removed)", alpha=0.6)
plt.plot(temperature_filtered[:1000], label="Filtered", alpha=0.9)

plt.title("Temperature Signal: Raw vs Filtered (Zoomed)")
plt.xlabel("Samples")
plt.ylabel("Temperature")
plt.legend()
plt.grid()
plt.show()

In [ ]:
import pandas as pd

pd.DataFrame({'EDA': eda_filtered}).to_csv('../Data/preprocessed_data/filtered/eda_filtered.csv', index=False)
pd.DataFrame({'BVP': bvp_filtered}).to_csv('../Data/preprocessed_data/filtered/bvp_filtered.csv', index=False)
pd.DataFrame({'HR': hr_filtered}).to_csv('../Data/preprocessed_data/filtered/hr_filtered.csv', index=False)
pd.DataFrame({'TEMP': temperature_filtered}).to_csv('../Data/preprocessed_data/filtered/temp_filtered.csv', index=False)

pd.DataFrame({'IBI': ibi_filtered}).to_csv('../Data/preprocessed_data/filtered/ibi_filtered.csv', index=False)

pd.DataFrame({'ECG': ecg_filtered}).to_csv('../Data/preprocessed_data/filtered/ecg_filtered_respiban.csv', index=False)
pd.DataFrame({'EDA': EDA_filtered}).to_csv('../Data/preprocessed_data/filtered/eda_filtered_respiban.csv', index=False)
pd.DataFrame({'EMG': emg_centered}).to_csv('../Data/preprocessed_data/filtered/emg_centered_respiban.csv', index=False)
pd.DataFrame({'RESP': RESPIRATION_filtered}).to_csv('../Data/preprocessed_data/filtered/resp_filtered_respiban.csv', index=False)
pd.DataFrame({'TEMP': temperature_filtered}).to_csv('../Data/preprocessed_data/filtered/temp_filtered_respiban.csv', index=False)

In [ ]:
import os, shutil

os.makedirs('../Data/preprocessed_data/filtered', exist_ok=True)
os.makedirs('../Data/preprocessed_data/normalized', exist_ok=True)
os.makedirs('../Data/preprocessed_data/windows_raw', exist_ok=True)
os.makedirs('../Data/preprocessed_data/windows_filtered', exist_ok=True)
os.makedirs('../Data/preprocessed_data/windows_normalized', exist_ok=True)

for f in os.listdir():
    if not f.endswith('.csv'):
        continue
    if f.startswith('windows_raw_'):
        shutil.move(f, f'../Data/preprocessed_data/windows_raw/{f}')
    elif f.startswith('windows_normalized_'):
        shutil.move(f, f'../Data/preprocessed_data/windows_normalized/{f}')
    elif f.startswith('windows_'):
        shutil.move(f, f'../Data/preprocessed_data/windows_filtered/{f}')
    elif f.startswith('normalized_'):
        shutil.move(f, f'../Data/preprocessed_data/normalized/{f}')
    elif f.endswith('_filtered.csv') or f.endswith('_filtered_respiban.csv'):
        shutil.move(f, f'../Data/preprocessed_data/filtered/{f}')

print("Done. Contents:")
for root, dirs, files in os.walk('../Data/preprocessed_data'):
    for f in files:
        print(os.path.join(root, f))